# RFM分析與客戶終身價值(CLV)預測

本notebook演示如何進行RFM分析並預測客戶終身價值。

## 學習目標
- 理解RFM (Recency, Frequency, Monetary) 分析
- 計算RFM指標並分配分數
- 基於RFM進行客戶分群
- 預測客戶終身價值(CLV)
- 識別高價值和流失風險客戶

In [ ]:
# 導入庫
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
from data_analysis_chatbots.clustering import RFMAnalyzer
from data_analysis_chatbots.marketing import CLVPredictor
from data_analysis_chatbots.visualization import Plotter

plt.style.use('seaborn-v0_8')
%matplotlib inline

print('✓ 庫導入成功')

## 1. 創建或載入交易數據

In [ ]:
# 生成範例交易數據
np.random.seed(42)

# 生成500個客戶的交易記錄
n_customers = 500
n_transactions = 5000

# 時間範圍: 過去1年
end_date = datetime(2024, 11, 1)
start_date = end_date - timedelta(days=365)

# 生成交易
transactions = []
for _ in range(n_transactions):
    customer_id = f'CUST{np.random.randint(1, n_customers+1):04d}'
    days_ago = np.random.randint(0, 365)
    transaction_date = end_date - timedelta(days=days_ago)
    amount = np.random.gamma(shape=2, scale=50)  # 偏斜分佈更真實
    
    transactions.append({
        'CustomerID': customer_id,
        'TransactionDate': transaction_date,
        'Amount': round(amount, 2)
    })

df = pd.DataFrame(transactions)
df['TransactionDate'] = pd.to_datetime(df['TransactionDate'])

print(f'✓ 生成 {len(df)} 筆交易記錄')
print(f'✓ 涵蓋 {df["CustomerID"].nunique()} 位客戶')
print(f'\n數據預覽:')
df.head(10)

In [ ]:
# 基本統計
print('交易數據統計:')
print(f'\n時間範圍: {df["TransactionDate"].min().date()} 到 {df["TransactionDate"].max().date()}')
print(f'平均交易金額: ${df["Amount"].mean():.2f}')
print(f'總交易金額: ${df["Amount"].sum():,.2f}')
print(f'\n金額分佈:')
print(df['Amount'].describe())

## 2. RFM分析

In [ ]:
# 初始化RFM分析器
rfm_analyzer = RFMAnalyzer(
    df=df,
    customer_id_col='CustomerID',
    date_col='TransactionDate',
    amount_col='Amount',
    reference_date=end_date
)

# 計算RFM指標
rfm_data = rfm_analyzer.calculate_rfm()

print('✓ RFM計算完成')
print(f'\nRFM數據預覽:')
rfm_data.head(10)

In [ ]:
# RFM指標統計
print('RFM指標統計:')
print(rfm_data[['Recency', 'Frequency', 'Monetary']].describe())

In [ ]:
# 可視化RFM分佈
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Recency分佈
axes[0].hist(rfm_data['Recency'], bins=30, color='skyblue', edgecolor='black')
axes[0].set_title('Recency分佈 (天)', fontsize=14, fontweight='bold')
axes[0].set_xlabel('最後購買距今天數')
axes[0].set_ylabel('客戶數')

# Frequency分佈
axes[1].hist(rfm_data['Frequency'], bins=30, color='lightgreen', edgecolor='black')
axes[1].set_title('Frequency分佈', fontsize=14, fontweight='bold')
axes[1].set_xlabel('交易次數')
axes[1].set_ylabel('客戶數')

# Monetary分佈
axes[2].hist(rfm_data['Monetary'], bins=30, color='lightcoral', edgecolor='black')
axes[2].set_title('Monetary分佈 ($)', fontsize=14, fontweight='bold')
axes[2].set_xlabel('總消費金額')
axes[2].set_ylabel('客戶數')

plt.tight_layout()
plt.show()

## 3. RFM分數與客戶分群

In [ ]:
# 分配RFM分數
rfm_scores = rfm_analyzer.assign_rfm_scores()

print('✓ RFM分數分配完成')
print(f'\nRFM分數預覽:')
print(rfm_scores[['CustomerID', 'Recency', 'Frequency', 'Monetary', 
                   'R_Score', 'F_Score', 'M_Score', 'RFM_Score']].head(10))

In [ ]:
# 進行客戶分群
segments = rfm_analyzer.segment_customers()

print('✓ 客戶分群完成')
print(f'\n分群分佈:')
print(segments['Segment'].value_counts())

In [ ]:
# 獲取分群摘要
segment_summary = rfm_analyzer.get_segment_summary()

print('\n客戶分群摘要:')
print(segment_summary.to_string())

In [ ]:
# 可視化分群分佈
plotter = Plotter()

plotter.plot_segment_distribution(
    segments=segments['Segment'],
    title='客戶分群分佈'
)

In [ ]:
# RFM散點圖 (按分群著色)
plt.figure(figsize=(14, 6))

plt.subplot(1, 2, 1)
for segment in segments['Segment'].unique():
    segment_data = segments[segments['Segment'] == segment]
    plt.scatter(segment_data['Recency'], segment_data['Monetary'], 
                label=segment, alpha=0.6, s=80)
plt.xlabel('Recency (天)', fontsize=12)
plt.ylabel('Monetary ($)', fontsize=12)
plt.title('Recency vs Monetary (按分群)', fontsize=14, fontweight='bold')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
for segment in segments['Segment'].unique():
    segment_data = segments[segments['Segment'] == segment]
    plt.scatter(segment_data['Frequency'], segment_data['Monetary'], 
                label=segment, alpha=0.6, s=80)
plt.xlabel('Frequency (次數)', fontsize=12)
plt.ylabel('Monetary ($)', fontsize=12)
plt.title('Frequency vs Monetary (按分群)', fontsize=14, fontweight='bold')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 4. 客戶終身價值(CLV)預測

In [ ]:
# 初始化CLV預測器
clv_predictor = CLVPredictor(
    discount_rate=0.1,  # 10% 年折扣率
    time_horizon_years=3  # 預測3年
)

# 基於RFM計算CLV
clv_results = clv_predictor.calculate_rfm_based_clv(
    rfm_df=segments,
    customer_lifespan_years=3
)

print('✓ CLV預測完成')
print(f'\nCLV預覽:')
print(clv_results[['CustomerID', 'Segment', 'Monetary', 'Predicted_CLV']].head(10))

In [ ]:
# CLV統計
clv_summary = clv_predictor.get_clv_summary(
    clv_df=clv_results,
    clv_col='Predicted_CLV',
    segment_col='Segment'
)

print('\nCLV摘要:')
print(f'總客戶數: {clv_summary["total_customers"]}')
print(f'總CLV: ${clv_summary["total_clv"]:,.2f}')
print(f'平均CLV: ${clv_summary["average_clv"]:.2f}')
print(f'中位數CLV: ${clv_summary["median_clv"]:.2f}')
print(f'最高CLV: ${clv_summary["max_clv"]:.2f}')
print(f'\n前10%客戶CLV閾值: ${clv_summary["top_10_percent_clv"]:.2f}')

if 'by_segment' in clv_summary:
    print('\n各分群CLV統計:')
    for segment, stats in clv_summary['by_segment'].items():
        print(f'\n{segment}:')
        print(f'  客戶數: {stats["count"]:.0f}')
        print(f'  總CLV: ${stats["sum"]:,.2f}')
        print(f'  平均CLV: ${stats["mean"]:.2f}')

In [ ]:
# CLV分群
clv_segments = clv_predictor.segment_by_clv(
    clv_df=clv_results,
    clv_col='Predicted_CLV',
    n_segments=4,
    labels=['低價值', '中等價值', '高價值', 'VIP']
)

print('\nCLV分群分佈:')
print(clv_segments['CLV_Segment'].value_counts())

In [ ]:
# 可視化CLV分佈
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# CLV直方圖
axes[0].hist(clv_segments['Predicted_CLV'], bins=50, color='steelblue', edgecolor='black')
axes[0].axvline(clv_segments['Predicted_CLV'].mean(), color='red', 
                linestyle='--', linewidth=2, label='平均值')
axes[0].axvline(clv_segments['Predicted_CLV'].median(), color='green', 
                linestyle='--', linewidth=2, label='中位數')
axes[0].set_xlabel('預測CLV ($)', fontsize=12)
axes[0].set_ylabel('客戶數', fontsize=12)
axes[0].set_title('CLV分佈', fontsize=14, fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# CLV分群箱型圖
clv_segments.boxplot(column='Predicted_CLV', by='CLV_Segment', ax=axes[1])
axes[1].set_xlabel('CLV分群', fontsize=12)
axes[1].set_ylabel('預測CLV ($)', fontsize=12)
axes[1].set_title('各CLV分群的價值分佈', fontsize=14, fontweight='bold')
plt.suptitle('')  # 移除默認標題

plt.tight_layout()
plt.show()

## 5. 高價值與風險客戶識別

In [ ]:
# 識別頂級客戶 (高CLV + Champions/Loyal)
top_customers = clv_segments[
    (clv_segments['CLV_Segment'] == 'VIP') & 
    (clv_segments['Segment'].isin(['Champions', 'Loyal Customers']))
].sort_values('Predicted_CLV', ascending=False)

print(f'✓ 識別出 {len(top_customers)} 位頂級客戶')
print(f'\n頂級客戶列表 (前10位):')
print(top_customers[['CustomerID', 'Segment', 'Monetary', 'Predicted_CLV']].head(10).to_string())
print(f'\n頂級客戶總CLV: ${top_customers["Predicted_CLV"].sum():,.2f}')

In [ ]:
# 識別流失風險客戶 (高CLV但At Risk / Can't Lose Them)
at_risk_high_value = clv_segments[
    (clv_segments['Predicted_CLV'] > clv_segments['Predicted_CLV'].quantile(0.75)) &
    (clv_segments['Segment'].isin(['At Risk', "Can't Lose Them", 'About to Sleep']))
].sort_values('Predicted_CLV', ascending=False)

print(f'\n⚠ 識別出 {len(at_risk_high_value)} 位高價值流失風險客戶')
print(f'\n流失風險高價值客戶 (前10位):')
print(at_risk_high_value[['CustomerID', 'Segment', 'Recency', 'Monetary', 'Predicted_CLV']].head(10).to_string())
print(f'\n潛在流失損失: ${at_risk_high_value["Predicted_CLV"].sum():,.2f}')

In [ ]:
# 潛力客戶 (Recent Customers 或 Potential Loyalists with 高CLV)
potential_stars = clv_segments[
    (clv_segments['CLV_Segment'].isin(['高價值', 'VIP'])) &
    (clv_segments['Segment'].isin(['Recent Customers', 'Potential Loyalists', 'Promising']))
].sort_values('Predicted_CLV', ascending=False)

print(f'\n⭐ 識別出 {len(potential_stars)} 位潛力之星客戶')
print(f'\n潛力客戶列表 (前10位):')
print(potential_stars[['CustomerID', 'Segment', 'Recency', 'Frequency', 'Predicted_CLV']].head(10).to_string())

## 6. 行動建議

In [ ]:
# 為不同群體制定策略
action_plan = {
    '頂級客戶': {
        '數量': len(top_customers),
        '總CLV': f"${top_customers['Predicted_CLV'].sum():,.2f}",
        '行動': [
            '提供VIP專屬服務和優先權',
            '邀請參加獨家活動',
            '提供個人化推薦和服務',
            '定期關懷和滿意度調查'
        ]
    },
    '流失風險高價值客戶': {
        '數量': len(at_risk_high_value),
        '潛在損失': f"${at_risk_high_value['Predicted_CLV'].sum():,.2f}",
        '行動': [
            '立即啟動挽回計劃',
            '提供特別優惠和折扣',
            '電話或專人聯繫了解需求',
            '解決可能的問題或不滿'
        ]
    },
    '潛力之星客戶': {
        '數量': len(potential_stars),
        '潛在價值': f"${potential_stars['Predicted_CLV'].sum():,.2f}",
        '行動': [
            '培養忠誠度計劃',
            '提供教育性內容和產品資訊',
            '鼓勵增加購買頻率',
            '交叉銷售和追加銷售'
        ]
    }
}

print('\n📋 行動計劃建議:\n')
print('='*80)
for group, plan in action_plan.items():
    print(f'\n{group.upper()}')
    print('-'*80)
    for key, value in plan.items():
        if isinstance(value, list):
            print(f'{key}:')
            for item in value:
                print(f'  • {item}')
        else:
            print(f'{key}: {value}')

## 7. 導出結果

In [ ]:
# 導出完整RFM-CLV分析結果
output_file = 'data/outputs/rfm_clv_analysis.csv'
clv_segments.to_csv(output_file, index=False)
print(f'✓ 完整分析結果已保存: {output_file}')

# 導出行動名單
top_customers.to_csv('data/outputs/top_customers.csv', index=False)
at_risk_high_value.to_csv('data/outputs/at_risk_high_value.csv', index=False)
potential_stars.to_csv('data/outputs/potential_stars.csv', index=False)

print('✓ 行動名單已保存:')
print('  - data/outputs/top_customers.csv')
print('  - data/outputs/at_risk_high_value.csv')
print('  - data/outputs/potential_stars.csv')

## 總結

### 關鍵發現
1. **客戶分群**: 識別出多個不同價值和參與度的客戶群體
2. **CLV預測**: 計算每位客戶的預測終身價值
3. **優先行動**: 識別需要立即關注的高價值客戶群

### 行動優先級
1. 🔴 **緊急**: 挽回流失風險高價值客戶
2. 🟡 **重要**: 維護頂級客戶關係
3. 🟢 **發展**: 培養潛力之星客戶

### 預期影響
- 減少高價值客戶流失
- 提升整體客戶終身價值
- 優化營銷資源配置